# Rectified journal-ready LDA notebook: fixed 5-topic model with strict keyword cleanup

This version keeps the v6 cation filter and v5 clear word-cloud labels, then tightens the journal-facing keyword display layer by removing OCR fragments and generic terms observed in the topic outputs, including `mtd`, `speci`, `eld`, `document`, `point`, and `key phrase`. Raw model terms remain exported for auditability.


# LDA topic modelling notebook: metaverse, blockchain, cryptocurrency, and digital assets

This redone Colab notebook keeps the LDA-first workflow and adds the requested journal-ready outputs:

- direct PDF upload in Google Colab using `RUN_MODE = "upload_pdfs"`;
- a clear conclusion on the recommended number of topics;
- a properly formatted topic-modelling table with normalised keywords;
- a top-20 normalised keyword visualisation across all documents;
- a normalised keyword co-occurrence network;
- improved high-resolution PNG, SVG, and PDF figures;
- a combined extracted-text file for all readable PDFs.

You can also run it from the generated `kumar_MV_corpus_dataset.jsonl.gz`, from a ZIP of PDFs, from Google Drive, or from an existing `/content/pdfs` folder.


In [ ]:
# 1. Install compatible Colab dependencies, then force a one-time restart.
# This fixes the pandas/numpy ABI error: "numpy.dtype size changed".
# Run this cell first. It will intentionally stop/restart the runtime once.
# After the runtime reconnects, run the notebook again from the top; this cell
# will detect the marker file and continue without reinstalling.
import os
import sys
import subprocess
from pathlib import Path

marker = Path("/content/.metaverse_lda_fixed_deps_installed")

if not marker.exists():
    packages = [
        "numpy==1.26.4",
        "pandas==2.2.2",
        "scipy==1.13.1",
        "gensim==4.3.3",
        "matplotlib==3.9.2",
        "seaborn==0.13.2",
        "pypdf==5.1.0",
        "pyLDAvis==3.4.1",
        "wordcloud==1.9.3",
        "tqdm==4.66.5",
        "openpyxl==3.1.5",
        "networkx==3.3",
        "nltk==3.9.1",
    ]
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        "--force-reinstall",
    ] + packages)
    marker.write_text("installed\n", encoding="utf-8")
    print("Compatible dependencies installed. Restarting runtime now to clear old numpy/pandas binaries...")
    os.kill(os.getpid(), 9)
else:
    print("Compatible dependencies already installed in this Colab runtime. Continuing to imports.")


In [ ]:
# 2. Imports and publication style.
from pathlib import Path
from collections import Counter
from itertools import combinations
import json
import hashlib
import gzip
import math
import os
import random
import re
import shutil
import textwrap
import zipfile

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from pypdf import PdfReader
from tqdm.auto import tqdm

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaMulticore
from gensim.models.phrases import Phrases, Phraser

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
for package in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.12)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
# 3. Configuration. Edit these values before running if needed.
RUN_MODE = "upload_pdfs"  # Options: "upload_pdfs", "upload_dataset", "upload_zip", "drive", or "existing".

PDF_DIR = Path("/content/pdfs")
CORPUS_DATASET_PATH = Path("/content/kumar_MV_corpus_dataset.jsonl.gz")
DRIVE_PDF_DIR = "/content/drive/MyDrive/kumar_MV"  # Used only when RUN_MODE = "drive".
OUTPUT_DIR = Path("/content/lda_outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
TEXT_DIR = OUTPUT_DIR / "extracted_text"

RANDOM_STATE = 42
FIGURE_FORMATS = ("png", "svg", "pdf")

# Text processing.
REMOVE_REFERENCES = True
DROP_DUPLICATE_TEXTS = True
MIN_TOKEN_LENGTH = 3
CHUNK_SIZE = 900
CHUNK_OVERLAP = 100
MIN_CHUNK_TOKENS = 150

# LDA modelling.
# Topic count is selected from diagnostics rather than fixed in advance.
# Set FORCE_TOPIC_COUNT to an integer only for an explicit sensitivity run.
FORCE_TOPIC_COUNT = None
TOPIC_RANGE = range(3, 9)
MAX_REPORTED_TOPICS = 8
LDA_PASSES = 12
LDA_ITERATIONS = 250
NO_BELOW = 4
NO_ABOVE = 0.45
KEEP_N = 60000
TOP_TERMS_FOR_DIAGNOSTICS = 15
DISPLAY_KEYWORDS_PER_TOPIC = 15
DISPLAY_CANDIDATE_TERMS = 200
ENFORCE_UNIQUE_DISPLAY_KEYWORDS = True
TOPIC_SELECTION_COHERENCE_TOLERANCE = 0.025
TOP_NORMALISED_KEYWORDS = 20
COOCCURRENCE_TOP_KEYWORDS = 60
COOCCURRENCE_MIN_EDGE_WEIGHT = 2
COOCCURRENCE_LABEL_COUNT = 24

# Optional robustness check. This adds runtime, so it is off by default.
RUN_STABILITY_CHECK = False
STABILITY_SEEDS = [11, 42, 73, 101]

# Use balanced training so long PDFs do not outweigh short PDFs during fitting.
# Set MAX_CHUNKS_PER_DOCUMENT_FOR_MODEL = None to train on every chunk.
MAX_CHUNKS_PER_DOCUMENT_FOR_MODEL = 50

# Domain stopwords should be used cautiously. Add only terms that are too broad
# for your research question, not terms that may define meaningful topics.
DOMAIN_STOPWORDS = set([
    # "metaverse",
    # "blockchain",
    # "cryptocurrency",
])

NORMALIZATION_ALIASES = {
    "crypto": "cryptocurrency",
    "cryptos": "cryptocurrency",
    "cryptocurrencies": "cryptocurrency",
    "nfts": "nft",
    "nonfungible": "nft",
    "non_fungible": "nft",
    "non_fungible_token": "nft",
    "non_fungible_tokens": "nft",
    "non-fungible": "nft",
    "non-fungible_token": "nft",
    "non-fungible_tokens": "nft",
    "crypto_currency": "cryptocurrency",
    "crypto_currencies": "cryptocurrency",
    "cryptoassets": "cryptoasset",
    "crypto_assets": "cryptoasset",
    "smart_contracts": "smart_contract",
    "digital_twins": "digital_twin",
    "virtual": "virtual_world",
    "virtual_worlds": "virtual_world",
    "virtual_world": "virtual_world",
    "virtual_words": "virtual_world",
    "decentralised": "decentralized",
    "decentralisation": "decentralization",
    "behaviour": "behavior",
    "behavioural": "behavioral",
    "privacy_preserving": "privacy_preserving",
    "classi_cation": "classification",
    "classifications": "classification",
}



# Strict journal-normalisation layer. These mappings reduce lexical fragmentation
# before dictionary creation and make labels more stable for publication use.
NORMALIZATION_ALIASES.update({
    "financial": "finance",
    "educational": "education",
    "educationally": "education",
    "students": "student",
    "universities": "university",
    "mtd": "method",
    "speci": "specific",
    "eld": "field",
    "financials": "finance",
    "finances": "finance",
    "fintech": "finance",
    "detect": "detection",
    "detected": "detection",
    "detecting": "detection",
    "detector": "detection",
    "detectors": "detection",
    "classi": "classification",
    "classify": "classification",
    "classified": "classification",
    "classifier": "classification",
    "classifiers": "classification",
    "predict": "prediction",
    "predicted": "prediction",
    "predicting": "prediction",
    "predictions": "prediction",
    "authenticate": "authentication",
    "authenticating": "authentication",
    "authenticated": "authentication",
    "authenticates": "authentication",
    "cyber_security": "cybersecurity",
    "cyber-security": "cybersecurity",
    "transactions": "transaction",
    "transacting": "transaction",
    "contracts": "contract",
    "protocols": "protocol",
    "tokens": "token",
    "nodes": "node",
    "markets": "market",
    "consumers": "consumer",
    "customers": "consumer",
    "games": "game",
    "gaming": "game",
    "players": "player",
    "attacks": "attack",
    "threats": "threat",
    "vulnerabilities": "vulnerability",
    "identities": "identity",
    "crypto_asset": "digital_asset",
    "cryptoasset": "digital_asset",
    "cryptoassets": "digital_asset",
    "crypto_assets": "digital_asset",
    "asset": "digital_asset",
    "assets": "digital_asset",
    "secure": "security",
    "securing": "security",
    "secured": "security",
    "arti_cial": "artificial",
    "arti_cial_intelligence": "artificial_intelligence",
})

CUSTOM_STOPWORDS = set(
    (
        "abstract article author copyright doi elsevier fig figs figure figures table tables "
        "introduction conclusion conclusions reference references appendix section "
        "result results method methods study studies paper research journal conference "
        "vol volume issue pp page pages et al using used use based proposed "
        "bmeoc ieee los que set sets"
    ).split()
)

DISPLAY_STOPWORDS = CUSTOM_STOPWORDS | set(
    (
        "analysis approach framework issue issues challenge challenges opportunity opportunities "
        "future implication implications proposed system systems technology technologies "
        "application applications environment environments data information user users "
        "feature features input output type types level levels medium media task tasks "
        "technique techniques schema scheme schemes device devices service services "
        "image images context dataset datasets communication communications"
    ).split()
)

# Extra filter used only for reporting-wide keyword frequency and co-occurrence outputs.
# These terms may remain available to LDA, but they are too broad for journal figures.
ADDITIONAL_KEYWORD_REPORTING_STOPWORDS = set(
    (
        "also another however therefore moreover furthermore hence thus may might could "
        "would should can cannot one two three many much various different related "
        "new value values time times process processes model models work works working development "
        "develop developed potential platform digital common general particular specific "
        "important key main major high low large small number amount part parts way ways "
        "area areas aspect aspects case cases example examples form forms level levels "
        "role roles need needs make makes made provide provides provided include includes including"
    ).split()
)



# Strict publication/reporting stopword layer. These words either carry little
# domain meaning or are OCR/non-English fragments observed in Round 1 outputs.
STRICT_TOPIC_NOISE_WORDS = set(
    (
        "well like without several create creates created term terms existing current present shown show due even since "
        "first still better although allows ability support multiple within thing things problem problems activity activities "
        "offer offers increase increases group groups physical project projects mechanism mechanisms concept concepts resource resources "
        "impact impacts word words text sample samples metric metrics era publication publications two three india bmeoc como una "
        "por para con los las del que esta este esto classi cation cations nftm mtd speci eld company companies "
        "many could would thus individual individuals veri tra md number numbers resident residents respondent respondents"
    ).split()
)
CUSTOM_STOPWORDS |= STRICT_TOPIC_NOISE_WORDS | set("bmeoc como una por para con los las del que esta este esto classi cation cations nftm mtd speci eld md".split())
DISPLAY_STOPWORDS |= STRICT_TOPIC_NOISE_WORDS | set(
    (
        "experience operation address design management framework frameworks service services solution solutions method methods "
        "application applications system systems technology technologies data information user users feature features input output "
        "level levels type types context communication communications document documents point points field fields "
        "keyword keywords phrase phrases keyphrase keyphrases empowered many could would thus individual individuals "
        "veri tra md number numbers resident residents respondent respondents"
    ).split()
)
DISPLAY_STOPWORDS |= ADDITIONAL_KEYWORD_REPORTING_STOPWORDS
STRICT_DISPLAY_EXCLUDE_TERMS = {
    "key phrase",
    "key phrases",
    "key_phrase",
    "key_phrases",
    "ai empowered",
    "ai_empowered",
    "veri",
    "tra",
    "many",
    "could",
    "would",
    "thus",
    "individual",
    "individuals",
    "number",
    "numbers",
    "resident",
    "residents",
    "respondent",
    "respondents",
    "digital medium",
    "digital media",
    "third level digital equality",
    "platform server",
    "platform servers",
    "md",
    "cp",
}
DISPLAY_STOPWORDS |= STRICT_DISPLAY_EXCLUDE_TERMS
ADDITIONAL_KEYWORD_REPORTING_STOPWORDS |= STRICT_TOPIC_NOISE_WORDS | DISPLAY_STOPWORDS | STRICT_DISPLAY_EXCLUDE_TERMS

# Single-word topic-table keywords must be recognisable domain concepts.
# This prevents generic words such as 'many', 'could', 'individual', or OCR fragments
# from appearing as topic descriptors while retaining meaningful terms such as 'token'.
DOMAIN_SINGLE_WORD_WHITELIST = set(
    (
        "algorithm attribute authentication attack avatar bitcoin blockchain business classification consumer contract "
        "creator cybersecurity defi decentralization detection education entity finance fraud game governance identity "
        "immersive jurisdiction legal market metaverse metaverses node nft ownership player privacy protocol "
        "property scam security smartcontract social swap token trust wallet world llm dao vr ar ai iot "
        "prediction infrastructure risk cryptocurrency encryption decryption access policy consumer protection intellectual property digital payments"
    ).split()
)


# Publication-facing consolidation for cryptography/access-control topic terms.
# These mappings affect displayed keywords and labels only; raw model terms remain
# exported separately in `raw_topic_terms` for auditability.
DISPLAY_TERM_CANONICAL_MAP = {
    "public key": "cryptographic key management",
    "private key": "cryptographic key management",
    "secret key": "cryptographic key management",
    "session key": "cryptographic key management",
    "symmetric key": "cryptographic key management",
    "key agreement": "cryptographic key management",
    "key generation": "cryptographic key management",
    "key management": "cryptographic key management",
    "cp abe": "attribute-based encryption",
    "cp-abe": "attribute-based encryption",
    "abe": "attribute-based encryption",
    "ciphertext policy attribute based encryption": "attribute-based encryption",
    "ciphertext-policy attribute-based encryption": "attribute-based encryption",
    "ciphertext policy abe": "attribute-based encryption",
    "attribute based encryption": "attribute-based encryption",
    "attribute-based encryption": "attribute-based encryption",
    "abac": "attribute-based access control",
    "abac md": "attribute-based access control",
    "attribute based access control": "attribute-based access control",
    "attribute-based access control": "attribute-based access control",
    "attribute": "attribute-based access control",
    "attributes": "attribute-based access control",
    "access policy": "access control policy",
    "policy attribute": "access control policy",
    "authentication scheme": "authentication protocol",
    "authentication phase": "authentication protocol",
    "authentication method": "authentication protocol",
    "encryption decryption": "encryption and decryption",
}

DISPLAY_TERM_FAMILY_MAP = {
    "cryptographic key management": "cryptographic_key_management",
    "attribute-based encryption": "attribute_based_encryption",
    "attribute-based access control": "attribute_based_access_control",
    "access control policy": "attribute_based_access_control",
    "authentication protocol": "authentication_protocol",
    "encryption and decryption": "encryption_decryption",
}


# Stricter journal-facing keyword rules. These remove source/watermark phrases,
# suppress broad single-word descriptors, and collapse repeated domain families.
DISPLAY_TERM_CANONICAL_MAP.update({
    "metaverses": "metaverse",
    "finance sector": "finance",
    "finance system": "finance",
    "finance systems": "finance",
    "finance service": "finance",
    "finance services": "finance",
    "finance innovation": "finance",
    "decentralized finance": "DeFi",
    "decentralised finance": "DeFi",
    "defi": "DeFi",
    "social medium": "social media",
    "grey economy": "grey economy",
    "gray economy": "grey economy",
    "cryptocurrency exchange": "cryptocurrency exchange",
    "bitcoin": "Bitcoin network",
    "bitcoin network": "Bitcoin network",
    "btc": "Bitcoin network",
    "btc trust": "trust",
    "msbc ctrust": "trust",
    "trust factor": "trust",
    "trust value": "trust",
    "trust badge": "trust",
    "detection rate": "fraud detection",
    "attribute key": "attribute-based encryption",
    "mc abe": "attribute-based encryption",
    "user authentication": "authentication protocol",
    "identity authentication": "authentication protocol",
    "encryption": "encryption and decryption",
    "decryption": "encryption and decryption",
    "privacy preserving": "privacy-preserving technology",
    "intellectual property": "intellectual property",
    "consumer protection": "consumer protection",
    "illicit fund": "illicit finance",
    "off chain": "off-chain transactions",
    "gas consumption": "gas fees",
    "digital payment": "digital payments",
})

DISPLAY_TERM_FAMILY_MAP.update({
    "metaverse": "metaverse",
    "finance": "finance",
    "DeFi": "defi",
    "defi": "defi",
    "social media": "social_media",
    "grey economy": "grey_economy",
    "cryptocurrency exchange": "cryptocurrency_exchange",
    "Bitcoin network": "bitcoin_network",
    "bitcoin network": "bitcoin_network",
    "trust": "trust",
    "fraud detection": "fraud_detection",
    "attribute-based encryption": "attribute_based_encryption",
    "authentication protocol": "authentication_protocol",
    "encryption and decryption": "encryption_decryption",
    "privacy-preserving technology": "privacy_preserving",
    "intellectual property": "intellectual_property",
    "consumer protection": "consumer_protection",
    "illicit finance": "illicit_finance",
    "off-chain transactions": "off_chain",
    "gas fees": "gas_fees",
    "digital payments": "digital_payments",
    "token": "token_protocol",
    "erc token": "token_protocol",
    "swap": "defi",
    "wallet": "crypto_wallet",
})

SOURCE_ARTIFACT_WORDS = set(
    "charles sturt university downloaded download authorized licensed licence license limited xplore utc january "
    "manning technological forecasting hasan elsevier springer mdpi ieee acm arxiv preprint".split()
)

NON_DOMAIN_SINGLE_WORDS = set(
    "world space business social economy entity policy market return layer block training pattern online people "
    "internet economic community review opportunity innovation enabler code characteristic graph law right behavior "
    "traditional content scalability interoperability dynamic ensuring urban sector service system medium".split()
)

NON_DOMAIN_EXACT_TERMS = {
    "downloaded charles sturt university",
    "downloaded january utc xplore",
    "authorized licensed limited charles",
    "sturt university",
    "charles sturt university",
    "surveyed work",
    "environment industry",
    "development attribute",
    "manning technological forecasting social",
    "internet hasan",
    "econ mico",
    "message call",
    "mapped value",
    "user server",
    "personhood score",
    "digital adaptability",
    "blockchain based",
    "real world",
    "real time",
    "finance sector",
    "finance system",
    "finance service",
    "finance innovation",
}

NON_DOMAIN_SUBSTRINGS = (
    "charles sturt",
    "downloaded",
    "authorized licensed",
    "licensed limited",
    "january utc",
    "xplore",
    "technological forecasting",
    "internet hasan",
    "econ mico",
)

# Controlled vocabulary for the top-20 domain-keyword figure. This prevents generic
# high-frequency words from replacing meaningful journal-facing concepts.
USE_CONTROLLED_DOMAIN_KEYWORDS_FOR_TOP20 = True
CONTROLLED_DOMAIN_KEYWORD_PATTERNS = [
    ("metaverse", r"\bmetaverse(?:s)?\b"),
    ("blockchain", r"\bblockchain(?:s)?\b"),
    ("security", r"\bsecurity\b|\bsecure\b|\bsecuring\b|\bcyber\s*security\b|\bcybersecurity\b"),
    ("NFT", r"\bNFTs?\b|\bnon[-\s]?fungible\s+tokens?\b"),
    ("transaction", r"\btransactions?\b|\btransacting\b"),
    ("decentralization", r"\bdecentralised\b|\bdecentralized\b|\bdecentralisation\b|\bdecentralization\b"),
    ("identity", r"\bidentity\b|\bidentities\b|\bidentification\b"),
    ("privacy", r"\bprivacy\b|\bprivacy[-\s]?preserving\b"),
    ("token", r"\btokens?\b|\btokeni[sz](?:e|ed|ation|ing)\b"),
    ("risk", r"\brisks?\b|\brisk[-\s]?management\b"),
    ("cryptocurrency", r"\bcrypto(?:currency|currencies)?\b|\bcryptocurrenc(?:y|ies)\b"),
    ("virtual reality", r"\bvirtual\s+reality\b|\bVR\b"),
    ("market", r"\bmarkets?\b|\bmarketplace(?:s)?\b"),
    ("trust", r"\btrust\b|\btrustworthiness\b|\btrusted\b"),
    ("algorithm", r"\balgorithms?\b"),
    ("governance", r"\bgovernance\b|\bregulat(?:e|ion|ory|ions)\b"),
    ("avatar", r"\bavatars?\b"),
    ("smart contract", r"\bsmart\s+contracts?\b"),
    ("virtual world", r"\bvirtual\s+world(?:s)?\b"),
    ("finance", r"\bfinance\b|\bfinancial\b|\bfintech\b"),
    ("digital asset", r"\bdigital\s+assets?\b|\bcrypto\s*assets?\b|\bcryptoassets?\b"),
    ("game", r"\bgames?\b|\bgaming\b|\bplayers?\b"),
    ("attack", r"\battacks?\b|\badversarial\s+attacks?\b"),
    ("threat", r"\bthreats?\b|\bcyber\s*threats?\b"),
    ("cybercrime", r"\bcyber\s*crime(?:s)?\b|\bcybercrime(?:s)?\b"),
    ("authentication", r"\bauthentication\b|\bauthenticat(?:e|ed|ing)\b"),
    ("access control", r"\baccess\s+control\b|\baccess\s+rights?\b"),
    ("Ethereum", r"\bethereum\b"),
    ("protocol", r"\bprotocols?\b"),
    ("DeFi", r"\bDeFi\b|\bdecentralised\s+finance\b|\bdecentralized\s+finance\b"),
    ("consumer", r"\bconsumers?\b|\bcustomers?\b"),
    ("social media", r"\bsocial\s+media\b|\bsocial\s+medium\b"),
    ("digital marketing", r"\bdigital\s+marketing\b|\badvertis(?:e|ing|ement|ements)\b"),
    ("fake news", r"\bfake\s+news\b|\bmisinformation\b|\bdisinformation\b"),
    ("machine learning", r"\bmachine\s+learning\b|\bML\b"),
    ("deep learning", r"\bdeep\s+learning\b"),
    ("detection", r"\bdetection\b|\bdetect(?:s|ed|ing)?\b|\bdetector(?:s)?\b"),
    ("classification", r"\bclassification\b|\bclassify\b|\bclassified\b|\bclassifier(?:s)?\b"),
    ("prediction", r"\bprediction\b|\bpredict(?:s|ed|ing)?\b"),
    ("augmented reality", r"\baugmented\s+reality\b|\bAR\b"),
    ("digital twin", r"\bdigital\s+twins?\b"),
]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

for directory in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, TEXT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def save_figure(fig, stem, tight=True):
    bbox = "tight" if tight else None
    paths = []
    for extension in FIGURE_FORMATS:
        path = FIGURE_DIR / f"{stem}.{extension}"
        fig.savefig(path, bbox_inches=bbox)
        paths.append(path)
    return paths


## Load PDFs

Use `RUN_MODE = "upload_pdfs"` to upload PDFs directly in Google Colab. You can select multiple PDF files in the upload dialog. For a faster repeat run, set `RUN_MODE = "upload_dataset"` and upload `kumar_MV_corpus_dataset.jsonl.gz`. For one compressed folder, use `RUN_MODE = "upload_zip"`. For Google Drive, use `RUN_MODE = "drive"` and set `DRIVE_PDF_DIR`.


In [ ]:
# 4. Upload or locate corpus inputs.
def clear_directory(path):
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

pdf_paths = []
if RUN_MODE == "upload_dataset":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No corpus dataset was uploaded.")
    dataset_candidates = [
        (name, data)
        for name, data in uploaded.items()
        if name.lower().endswith((".jsonl", ".jsonl.gz"))
    ]
    if not dataset_candidates:
        uploaded_names = ", ".join(uploaded.keys())
        raise RuntimeError(
            "RUN_MODE is 'upload_dataset', so upload the generated "
            "kumar_MV_corpus_dataset.jsonl.gz file, not the notebook, PDFs, or ZIP. "
            f"Received: {uploaded_names}"
        )
    dataset_name, dataset_bytes = dataset_candidates[0]
    CORPUS_DATASET_PATH = Path("/content") / Path(dataset_name).name
    CORPUS_DATASET_PATH.write_bytes(dataset_bytes)
    print(f"Uploaded corpus dataset: {CORPUS_DATASET_PATH}")
elif RUN_MODE in {"upload_pdfs", "upload_zip"}:
    from google.colab import files
    clear_directory(PDF_DIR)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No PDFs or ZIP file were uploaded.")
    for upload_name, upload_bytes in uploaded.items():
        upload_path = Path(upload_name)
        if upload_path.suffix.lower() == ".zip":
            temp_zip = Path("/content") / upload_path.name
            temp_zip.write_bytes(upload_bytes)
            with zipfile.ZipFile(temp_zip, "r") as archive:
                archive.extractall(PDF_DIR)
        elif upload_path.suffix.lower() == ".pdf":
            (PDF_DIR / upload_path.name).write_bytes(upload_bytes)
        else:
            print(f"Skipping unsupported upload: {upload_name}")
elif RUN_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PDF_DIR = Path(DRIVE_PDF_DIR)
elif RUN_MODE == "existing":
    PDF_DIR = Path(PDF_DIR)
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

if RUN_MODE != "upload_dataset":
    pdf_paths = sorted(PDF_DIR.rglob("*.pdf"))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found under {PDF_DIR}")

    print(f"Found {len(pdf_paths)} PDFs under {PDF_DIR}")
    for path in pdf_paths[:10]:
        print(" -", path.name)
    if len(pdf_paths) > 10:
        print(f"... plus {len(pdf_paths) - 10} more")


## Extract text and audit the corpus

This cell extracts text from all PDFs and writes a corpus manifest. Review warnings for unusually long PDFs, low-text PDFs, or extraction errors before interpreting topics.

Exact duplicate extracted texts are flagged. By default, duplicate texts are excluded from modelling after the first copy, while every PDF remains listed in the manifest.


In [ ]:
# 5. PDF extraction and manifest.
def slugify(value, limit=60):
    value = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_").lower()
    return (value[:limit] or "document")

def compact_text(text, limit=220):
    return re.sub(r"\s+", " ", text).strip()[:limit]

def strip_references(text):
    if not REMOVE_REFERENCES:
        return text
    lower = text.lower()
    pattern = re.compile(r"\n\s*(references|bibliography|works cited)\s*\n")
    matches = list(pattern.finditer(lower))
    for match in matches:
        if match.start() > len(text) * 0.45:
            return text[:match.start()]
    return text

def extract_pdf(pdf_path, doc_id):
    row = {
        "doc_id": doc_id,
        "filename": pdf_path.name,
        "relative_path": str(pdf_path.relative_to(PDF_DIR)) if PDF_DIR in pdf_path.parents else pdf_path.name,
        "read_status": "ok",
        "read_error": "",
        "is_encrypted": False,
        "text_sha256": "",
        "pages": 0,
        "extracted_chars": 0,
        "extracted_words": 0,
        "low_text_pages": 0,
        "first_text_preview": "",
    }
    pages = []
    try:
        reader = PdfReader(str(pdf_path))
        row["is_encrypted"] = bool(getattr(reader, "is_encrypted", False))
        if row["is_encrypted"]:
            try:
                reader.decrypt("")
            except Exception:
                pass
        row["pages"] = len(reader.pages)
        for page in reader.pages:
            page_text = page.extract_text() or ""
            pages.append(page_text)
        text = "\n\n".join(pages)
        row["extracted_chars"] = len(text)
        row["extracted_words"] = len(re.findall(r"[A-Za-z][A-Za-z'-]+", text))
        row["text_sha256"] = hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()
        row["low_text_pages"] = sum(1 for page_text in pages if len(page_text) < 100)
        row["first_text_preview"] = compact_text(text)
        return row, text, strip_references(text)
    except Exception as exc:
        row["read_status"] = "error"
        row["read_error"] = f"{type(exc).__name__}: {exc}"
        return row, "", ""

def read_corpus_dataset(dataset_path):
    with open(dataset_path, "rb") as signature_handle:
        signature = signature_handle.read(2)
    opener = gzip.open if signature == b"\x1f\x8b" else open
    rows = []
    try:
        with opener(dataset_path, "rt", encoding="utf-8-sig") as handle:
            for line_number, line in enumerate(handle, start=1):
                if line.strip():
                    rows.append(json.loads(line))
    except (UnicodeDecodeError, json.JSONDecodeError, OSError) as exc:
        raise RuntimeError(
            "The uploaded corpus dataset could not be read. Upload "
            "kumar_MV_corpus_dataset.jsonl.gz from the output/data folder. "
            "If you want to upload PDFs or a ZIP instead, set RUN_MODE = 'upload_zip'."
        ) from exc
    if not rows:
        raise RuntimeError("The uploaded corpus dataset was empty.")
    return rows

records = []
raw_texts = {}
all_extracted_texts = {}
if RUN_MODE == "upload_dataset":
    dataset_rows = read_corpus_dataset(CORPUS_DATASET_PATH)
    for index, record in enumerate(dataset_rows, start=1):
        text = record.get("text", "") or ""
        analysis_text = strip_references(text)
        doc_id = record.get("doc_id") or f"D{index:03d}_{slugify(record.get('filename', f'document_{index}'))}"
        row = {
            "doc_id": doc_id,
            "filename": record.get("filename", f"document_{index}.pdf"),
            "relative_path": record.get("relative_path", record.get("filename", "")),
            "read_status": "ok" if text.strip() else "empty",
            "read_error": "",
            "is_encrypted": False,
            "text_sha256": record.get("text_sha256") or hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest(),
            "pages": int(record.get("pages", 0) or 0),
            "extracted_chars": len(text),
            "extracted_words": int(record.get("extracted_words", 0) or len(re.findall(r"[A-Za-z][A-Za-z'-]+", text))),
            "low_text_pages": np.nan,
            "first_text_preview": compact_text(text),
        }
        records.append(row)
        if text.strip():
            all_extracted_texts[doc_id] = text
        if analysis_text.strip():
            raw_texts[doc_id] = analysis_text
else:
    for index, pdf_path in enumerate(tqdm(pdf_paths, desc="Extracting PDFs"), start=1):
        doc_id = f"D{index:03d}_{slugify(pdf_path.stem)}"
        row, extracted_text, analysis_text = extract_pdf(pdf_path, doc_id)
        records.append(row)
        if row["read_status"] == "ok" and extracted_text.strip():
            all_extracted_texts[doc_id] = extracted_text
        if row["read_status"] == "ok" and analysis_text.strip():
            raw_texts[doc_id] = analysis_text

manifest = pd.DataFrame(records)
manifest["char_per_page"] = manifest["extracted_chars"] / manifest["pages"].replace(0, np.nan)
manifest["long_document_flag"] = manifest["pages"] > manifest["pages"].quantile(0.90)
manifest["low_text_flag"] = manifest["extracted_chars"] < 1000
manifest["duplicate_text_flag"] = manifest.duplicated("text_sha256", keep=False) & manifest["text_sha256"].astype(bool)

if DROP_DUPLICATE_TEXTS:
    duplicate_doc_ids = manifest.loc[
        manifest.duplicated("text_sha256", keep="first") & manifest["text_sha256"].astype(bool),
        "doc_id"
    ].tolist()
    for duplicate_doc_id in duplicate_doc_ids:
        raw_texts.pop(duplicate_doc_id, None)
else:
    duplicate_doc_ids = []

manifest["included_for_modelling"] = manifest["doc_id"].isin(raw_texts.keys())
manifest_path = TABLE_DIR / "corpus_manifest.csv"
manifest.to_csv(manifest_path, index=False)

def write_combined_extracted_text(manifest_df, extracted_texts, path):
    sections = []
    export_rows = []
    for row in manifest_df.sort_values("doc_id").itertuples(index=False):
        text = extracted_texts.get(row.doc_id, "")
        export_rows.append({
            "doc_id": row.doc_id,
            "filename": row.filename,
            "included_in_text_export": bool(text.strip()),
            "exported_chars": len(text),
            "exported_words": len(re.findall(r"[A-Za-z][A-Za-z'-]+", text)),
        })
        if not text.strip():
            continue
        header = (
            f"\n\n{'=' * 100}\n"
            f"DOC_ID: {row.doc_id}\n"
            f"FILENAME: {row.filename}\n"
            f"PAGES: {row.pages}\n"
            f"EXTRACTED_WORDS: {row.extracted_words}\n"
            f"{'=' * 100}\n\n"
        )
        sections.append(header + text.strip())
    path.write_text("\n".join(sections).strip() + "\n", encoding="utf-8")
    export_df = pd.DataFrame(export_rows)
    export_df.to_csv(TABLE_DIR / "extracted_text_export_manifest.csv", index=False)
    return path, export_df

combined_text_path, text_export_manifest = write_combined_extracted_text(
    manifest,
    all_extracted_texts,
    TEXT_DIR / "all_extracted_text_by_pdf.txt",
)

display(manifest[[
    "doc_id", "filename", "pages", "extracted_words", "read_status",
    "long_document_flag", "low_text_flag", "duplicate_text_flag", "included_for_modelling"
]].sort_values("pages", ascending=False).head(12))

print(f"Readable PDFs: {(manifest['read_status'] == 'ok').sum()} / {len(manifest)}")
print(f"Exact duplicate texts excluded from modelling: {len(duplicate_doc_ids)}")
print(f"Manifest written to: {manifest_path}")
print(f"Combined extracted text written to: {combined_text_path}")


In [ ]:
# 6. Tokenisation, lemmatisation, phrase detection, and chunking.
lemmatizer = WordNetLemmatizer()
base_stopwords = set(stopwords.words("english"))
all_stopwords = base_stopwords | CUSTOM_STOPWORDS | DOMAIN_STOPWORDS

token_re = re.compile(r"[a-z][a-z_'-]{2,}")

MULTIWORD_ALIAS_PATTERNS = [
    (r"\bnon[\s-]?fungible\s+tokens?\b", " nft "),
    (r"\bnfts\b", " nft "),
    (r"\bcrypto\b", " cryptocurrency "),
    (r"\bcrypto[\s-]?currencies\b", " cryptocurrency "),
    (r"\bcrypto[\s-]?currency\b", " cryptocurrency "),
    (r"\bcrypto[\s-]?assets\b", " cryptoasset "),
    (r"\bsmart\s+contracts\b", " smart_contract "),
    (r"\bdigital\s+twins\b", " digital_twin "),
    (r"\bvirtual\s+worlds?\b", " virtual_world "),
    (r"\bvirtual\s+words?\b", " virtual_world "),
]

BROKEN_WORD_REPAIRS = [
    (r"\bclassi\s+cation\b", "classification"),
    (r"\bclassi\s+cations\b", "classification"),
    (r"\bclassi\s+ed\b", "classified"),
    (r"\bclassi\s+er\b", "classifier"),
    (r"\bclassi\s+ers\b", "classifiers"),
    (r"\bidenti\s+cation\b", "identification"),
    (r"\bspeci\s+c\b", "specific"),
    (r"\bfi\s+eld\b", "field"),
    (r"\bidenti\s+ed\b", "identified"),
    (r"\bveri\s+cation\b", "verification"),
    (r"\bcerti\s+cate\b", "certificate"),
    (r"\bcerti\s+cates\b", "certificates"),
    (r"\barti\s+cial\b", "artificial"),
    (r"\bcon\s+dential\b", "confidential"),
    (r"\bcon\s+dentiality\b", "confidentiality"),
    (r"\bef\s+cient\b", "efficient"),
    (r"\bef\s+ciency\b", "efficiency"),
    (r"\bspeci\s+c\b", "specific"),
    (r"\bspeci\s+cally\b", "specifically"),
    (r"\bsigni\s+cant\b", "significant"),
    (r"\bsigni\s+cance\b", "significance"),
    (r"\bbene\s+t\b", "benefit"),
    (r"\bbene\s+ts\b", "benefits"),
    (r"\bde\s+nition\b", "definition"),
    (r"\bde\s+ned\b", "defined"),
]

normalization_map = pd.DataFrame(
    [{"source": source, "normalised": target} for source, target in NORMALIZATION_ALIASES.items()]
    + [{"source": pattern, "normalised": target.strip()} for pattern, target in MULTIWORD_ALIAS_PATTERNS]
    + [{"source": pattern, "normalised": target} for pattern, target in BROKEN_WORD_REPAIRS]
)
normalization_map.to_csv(TABLE_DIR / "normalization_map.csv", index=False)

def apply_text_aliases(text):
    for pattern, replacement in BROKEN_WORD_REPAIRS:
        text = re.sub(pattern, replacement, text)
    for pattern, replacement in MULTIWORD_ALIAS_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

def normalize_token(token):
    token = token.replace("-", "_").strip("_'")
    token = NORMALIZATION_ALIASES.get(token, token)
    lemma = lemmatizer.lemmatize(token)
    lemma = NORMALIZATION_ALIASES.get(lemma, lemma)
    return lemma

def collapse_adjacent_duplicates(tokens):
    collapsed = []
    for token in tokens:
        if not collapsed or collapsed[-1] != token:
            collapsed.append(token)
    return collapsed

def clean_for_tokens(text):
    text = text.lower()
    text = apply_text_aliases(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\bdoi\s*:\s*\S+", " ", text)
    text = re.sub(r"\b[a-z]\s*/\s?[a-z]\b", " ", text)
    text = re.sub(r"\b\d+(?:\.\d+)?\b", " ", text)
    text = re.sub(r"[^a-z_\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize(text):
    clean = clean_for_tokens(text)
    tokens = []
    for token in token_re.findall(clean):
        token = token.strip("_'-")
        if len(token) < MIN_TOKEN_LENGTH:
            continue
        if token in all_stopwords:
            continue
        lemma = normalize_token(token)
        if lemma in all_stopwords or len(lemma) < MIN_TOKEN_LENGTH:
            continue
        tokens.append(lemma)
    return collapse_adjacent_duplicates(tokens)

doc_token_rows = []
for doc_id, text in tqdm(raw_texts.items(), desc="Tokenising"):
    filename = manifest.loc[manifest["doc_id"] == doc_id, "filename"].iloc[0]
    tokens = tokenize(text)
    doc_token_rows.append({"doc_id": doc_id, "filename": filename, "tokens": tokens, "token_count": len(tokens)})

doc_tokens = pd.DataFrame(doc_token_rows)
if doc_tokens.empty:
    raise RuntimeError("No readable text was tokenised. Check the PDF extraction results.")

phrase_source = doc_tokens["tokens"].tolist()
bigram = Phrases(phrase_source, min_count=5, threshold=12, delimiter="_")
bigram_model = Phraser(bigram)
trigram = Phrases([bigram_model[tokens] for tokens in phrase_source], min_count=5, threshold=12, delimiter="_")
trigram_model = Phraser(trigram)

doc_tokens["tokens"] = doc_tokens["tokens"].apply(
    lambda tokens: collapse_adjacent_duplicates([
        normalize_token(token)
        for token in trigram_model[bigram_model[tokens]]
        if normalize_token(token) not in all_stopwords
    ])
)
doc_tokens["token_count"] = doc_tokens["tokens"].apply(len)


def readable_keyword(term):
    return term.replace("_", " ").strip()

def keyword_display_key(term):
    return re.sub(r"\s+", " ", readable_keyword(term).lower()).strip()

keyword_reporting_stopwords = base_stopwords | DISPLAY_STOPWORDS | ADDITIONAL_KEYWORD_REPORTING_STOPWORDS
pd.DataFrame(
    {"keyword_reporting_stopword": sorted(keyword_reporting_stopwords)}
).to_csv(TABLE_DIR / "keyword_reporting_stopwords.csv", index=False)

def keyword_is_displayable(term):
    key = keyword_display_key(term)
    words = key.split()
    if not words:
        return False
    if term in keyword_reporting_stopwords or key in keyword_reporting_stopwords:
        return False
    if key in STRICT_DISPLAY_EXCLUDE_TERMS or term in STRICT_DISPLAY_EXCLUDE_TERMS:
        return False
    if any(word in STRICT_TOPIC_NOISE_WORDS for word in words):
        return False
    if all(word in keyword_reporting_stopwords for word in words):
        return False
    return True

keyword_token_counter = Counter()
keyword_document_counter = Counter()
keyword_doc_rows = []
for row in doc_tokens.itertuples(index=False):
    display_terms = [keyword_display_key(token) for token in row.tokens if keyword_is_displayable(token)]
    keyword_token_counter.update(display_terms)
    keyword_document_counter.update(set(display_terms))
    for term, count in Counter(display_terms).items():
        keyword_doc_rows.append({
            "doc_id": row.doc_id,
            "filename": row.filename,
            "normalised_keyword": term,
            "document_token_count": count,
        })

normalised_keyword_by_document = pd.DataFrame(keyword_doc_rows)
normalised_keyword_frequencies = pd.DataFrame([
    {
        "normalised_keyword": term,
        "token_frequency": int(keyword_token_counter[term]),
        "document_frequency": int(keyword_document_counter[term]),
    }
    for term in keyword_token_counter
]).sort_values(["document_frequency", "token_frequency", "normalised_keyword"], ascending=[False, False, True])
normalised_keyword_frequencies["document_share"] = normalised_keyword_frequencies["document_frequency"] / len(doc_tokens)
normalised_keyword_frequencies.to_csv(TABLE_DIR / "normalised_keyword_frequencies.csv", index=False)
normalised_keyword_by_document.to_csv(TABLE_DIR / "normalised_keyword_by_document.csv", index=False)

def compute_controlled_domain_keyword_frequencies(text_by_doc):
    rows = []
    for label, pattern in CONTROLLED_DOMAIN_KEYWORD_PATTERNS:
        regex = re.compile(pattern, flags=re.IGNORECASE)
        document_frequency = 0
        token_frequency = 0
        example_filenames = []
        for doc_id, text in text_by_doc.items():
            matches = regex.findall(text or "")
            if matches:
                document_frequency += 1
                token_frequency += len(matches)
                if len(example_filenames) < 3:
                    filename = manifest.loc[manifest["doc_id"] == doc_id, "filename"]
                    example_filenames.append(filename.iloc[0] if not filename.empty else doc_id)
        if document_frequency:
            rows.append({
                "normalised_keyword": label,
                "document_frequency": int(document_frequency),
                "token_frequency": int(token_frequency),
                "example_pdfs": " | ".join(example_filenames),
            })
    if not rows:
        return pd.DataFrame(columns=["normalised_keyword", "document_frequency", "token_frequency", "example_pdfs"])
    return pd.DataFrame(rows).sort_values(
        ["document_frequency", "token_frequency", "normalised_keyword"],
        ascending=[False, False, True],
    )

domain_keyword_frequencies = compute_controlled_domain_keyword_frequencies(raw_texts)
domain_keyword_frequencies.to_csv(TABLE_DIR / "domain_keyword_frequencies.csv", index=False)

if USE_CONTROLLED_DOMAIN_KEYWORDS_FOR_TOP20 and not domain_keyword_frequencies.empty:
    top20_normalised_keywords = domain_keyword_frequencies.head(TOP_NORMALISED_KEYWORDS).copy()
else:
    top20_normalised_keywords = normalised_keyword_frequencies.head(TOP_NORMALISED_KEYWORDS).copy()

top20_normalised_keywords.to_csv(TABLE_DIR / "top_20_normalised_keywords.csv", index=False)

def chunk_tokens(tokens, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, min_tokens=MIN_CHUNK_TOKENS):
    if chunk_size <= overlap:
        raise ValueError("CHUNK_SIZE must be larger than CHUNK_OVERLAP.")
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(tokens), step):
        chunk = tokens[start:start + chunk_size]
        if len(chunk) >= min_tokens:
            chunks.append((start, chunk))
    return chunks

chunk_rows = []
for row in doc_tokens.itertuples(index=False):
    for chunk_number, (start_token, chunk) in enumerate(chunk_tokens(row.tokens), start=1):
        chunk_rows.append({
            "chunk_id": f"{row.doc_id}_C{chunk_number:03d}",
            "doc_id": row.doc_id,
            "filename": row.filename,
            "chunk_number": chunk_number,
            "start_token": start_token,
            "token_count": len(chunk),
            "tokens": chunk,
            "snippet": " ".join(chunk[:80]),
        })

chunks = pd.DataFrame(chunk_rows)
if chunks.empty:
    raise RuntimeError("No chunks were created. Lower MIN_CHUNK_TOKENS or inspect extraction quality.")

token_audit = doc_tokens.drop(columns=["tokens"]).merge(
    chunks.groupby("doc_id").size().rename("chunk_count"),
    on="doc_id",
    how="left",
)
token_audit.to_csv(TABLE_DIR / "tokenisation_audit.csv", index=False)
display(token_audit.sort_values("token_count", ascending=False).head(12))
print(f"Created {len(chunks)} analysis chunks from {len(doc_tokens)} PDFs.")
print(f"Normalised keyword frequency table written to: {TABLE_DIR / 'normalised_keyword_frequencies.csv'}")
display(top20_normalised_keywords)


In [ ]:
# 7. Build dictionary and model corpus.
def balanced_model_chunks(chunks_df):
    if MAX_CHUNKS_PER_DOCUMENT_FOR_MODEL is None:
        return chunks_df.copy()
    sampled = []
    for _, group in chunks_df.groupby("doc_id", sort=False):
        n = min(len(group), MAX_CHUNKS_PER_DOCUMENT_FOR_MODEL)
        sampled.append(group.sample(n=n, random_state=RANDOM_STATE))
    return pd.concat(sampled, ignore_index=True)

model_chunks = balanced_model_chunks(chunks)

dictionary = Dictionary(model_chunks["tokens"].tolist())
dictionary.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE, keep_n=KEEP_N)
dictionary.compactify()

model_bows = [dictionary.doc2bow(tokens) for tokens in model_chunks["tokens"]]
model_keep = [index for index, bow in enumerate(model_bows) if bow]
model_chunks = model_chunks.iloc[model_keep].reset_index(drop=True)
model_bows = [model_bows[index] for index in model_keep]
model_texts = model_chunks["tokens"].tolist()

all_bows = [dictionary.doc2bow(tokens) for tokens in chunks["tokens"]]
all_keep = [index for index, bow in enumerate(all_bows) if bow]
scored_chunks = chunks.iloc[all_keep].reset_index(drop=True)
all_bows = [all_bows[index] for index in all_keep]

if not model_bows:
    raise RuntimeError("Dictionary filtering removed all model terms. Lower NO_BELOW or raise NO_ABOVE.")

vocab_path = TABLE_DIR / "lda_vocabulary.csv"
pd.DataFrame(
    [{"term_id": term_id, "term": term} for term_id, term in dictionary.items()]
).to_csv(vocab_path, index=False)

print(f"Vocabulary size: {len(dictionary):,}")
print(f"Model-training chunks: {len(model_chunks):,}")
print(f"Scored chunks: {len(scored_chunks):,}")
print(f"Vocabulary written to: {vocab_path}")


## Select the number of topics

The next cell fits LDA models across `TOPIC_RANGE` and selects the topic count using a coherence-first metric rule. Inspect the summary table and `pyLDAvis` after the run; use `FORCE_TOPIC_COUNT` only for a deliberate sensitivity run.


In [ ]:
# 8. Model selection by coherence and log perplexity.
workers = max(1, (os.cpu_count() or 2) - 1)
models = {}
model_rows = []
model_token_count = int(sum(count for bow in model_bows for _, count in bow))

candidate_topic_counts = sorted(set(int(k) for k in TOPIC_RANGE) | ({int(FORCE_TOPIC_COUNT)} if FORCE_TOPIC_COUNT else set()))
if not candidate_topic_counts:
    raise RuntimeError("No candidate topic counts were configured.")

for topic_count in tqdm(candidate_topic_counts, desc="Fitting candidate LDA models"):
    model = LdaMulticore(
        corpus=model_bows,
        id2word=dictionary,
        num_topics=topic_count,
        random_state=RANDOM_STATE,
        chunksize=2000,
        passes=LDA_PASSES,
        iterations=LDA_ITERATIONS,
        alpha="asymmetric",
        eta="auto",
        workers=workers,
        eval_every=None,
    )
    coherence = CoherenceModel(
        model=model,
        texts=model_texts,
        dictionary=dictionary,
        coherence="c_v",
    ).get_coherence()
    log_likelihood_bound = float(model.bound(model_bows))
    per_word_log_likelihood_bound = log_likelihood_bound / max(1, model_token_count)
    log_perplexity = float(model.log_perplexity(model_bows))
    perplexity = float(np.exp2(-log_perplexity))
    models[topic_count] = model
    model_rows.append({
        "topics": topic_count,
        "coherence_cv": coherence,
        "log_likelihood_bound": log_likelihood_bound,
        "per_word_log_likelihood_bound": per_word_log_likelihood_bound,
        "log_perplexity": log_perplexity,
        "perplexity": perplexity,
        "model_token_count": model_token_count,
    })

model_selection = pd.DataFrame(model_rows)
model_selection["coherence_rank"] = model_selection["coherence_cv"].rank(method="min", ascending=False).astype(int)
model_selection.to_csv(TABLE_DIR / "model_selection.csv", index=False)
display(model_selection)

coherence_best_row = model_selection.sort_values(["coherence_cv", "topics"], ascending=[False, True]).iloc[0]
coherence_best_k = int(coherence_best_row["topics"])

if FORCE_TOPIC_COUNT is not None:
    best_k = int(FORCE_TOPIC_COUNT)
    if best_k not in models:
        raise RuntimeError(f"FORCE_TOPIC_COUNT={best_k} was not fitted. Check TOPIC_RANGE and FORCE_TOPIC_COUNT.")
    topic_selection_decision = "forced sensitivity-run topic count"
    topic_selection_reason = (
        f"The model was explicitly forced to {best_k} topics. "
        f"The coherence-optimal candidate in this diagnostic run was {coherence_best_k} topics."
    )
else:
    eligible = model_selection[model_selection["topics"] <= MAX_REPORTED_TOPICS].copy()
    if eligible.empty:
        eligible = model_selection.copy()
    best_metric_row = eligible.sort_values(
        ["coherence_cv", "perplexity", "topics"],
        ascending=[False, True, True],
    ).iloc[0]
    best_k = int(best_metric_row["topics"])
    topic_selection_decision = "metric-optimised coherence-first solution"
    topic_selection_reason = (
        f"The selected model has the highest c_v coherence among eligible candidates "
        f"(ties are resolved by lower perplexity, then fewer topics). Perplexity and "
        f"log-likelihood are reported as supporting diagnostics rather than as the sole decision rule."
    )

best_model = models[best_k]
selected_model_metrics = model_selection.loc[model_selection["topics"] == best_k].copy()
selected_model_metrics.to_csv(TABLE_DIR / "selected_model_metrics.csv", index=False)
best_row = selected_model_metrics.iloc[0]

print(f"Selected topic count: {best_k}")
print(topic_selection_reason)
print(
    "Selected model metrics: "
    f"c_v coherence={best_row['coherence_cv']:.3f}, "
    f"log likelihood bound={best_row['log_likelihood_bound']:.1f}, "
    f"perplexity={best_row['perplexity']:.3f}"
)

topic_count_recommendation = pd.DataFrame([{
    "recommended_topic_count": best_k,
    "decision_rule": topic_selection_decision,
    "selected_coherence_cv": float(best_row["coherence_cv"]),
    "selected_perplexity": float(best_row["perplexity"]),
    "selected_log_likelihood_bound": float(best_row["log_likelihood_bound"]),
    "coherence_optimal_topic_count": coherence_best_k,
    "coherence_optimal_cv": float(coherence_best_row["coherence_cv"]),
    "coherence_tolerance": TOPIC_SELECTION_COHERENCE_TOLERANCE,
    "conclusion": f"Use {best_k} topics for the main journal model.",
}])
topic_count_recommendation.to_csv(TABLE_DIR / "topic_count_recommendation.csv", index=False)

topic_count_conclusion_text = f"""# Topic Count Recommendation

## Conclusion

Use **{best_k} topics** for the main journal model.

## Rationale

{topic_selection_reason}

Selected model metrics: c_v coherence {float(best_row['coherence_cv']):.3f}, perplexity {float(best_row['perplexity']):.3f}, and log-likelihood bound {float(best_row['log_likelihood_bound']):.1f}.

The coherence-optimal candidate in the diagnostic range {candidate_topic_counts} was {coherence_best_k} topics with c_v coherence {float(coherence_best_row['coherence_cv']):.3f}. Coherence is reported as a diagnostic, not as an automatic instruction to keep adding topics.
"""
(OUTPUT_DIR / "topic_count_conclusion.md").write_text(topic_count_conclusion_text, encoding="utf-8")
print(topic_count_conclusion_text)


In [ ]:
# 8b. Optional seed-stability audit for the selected topic count.
def top_term_sets(model, topic_count, topn=TOP_TERMS_FOR_DIAGNOSTICS):
    return [set(term for term, _ in model.show_topic(topic_index, topn=topn)) for topic_index in range(topic_count)]

stability_summary = pd.DataFrame()
if RUN_STABILITY_CHECK:
    baseline_terms = top_term_sets(best_model, best_k)
    stability_rows = []
    for seed in tqdm(STABILITY_SEEDS, desc="Running stability check"):
        stability_model = LdaMulticore(
            corpus=model_bows,
            id2word=dictionary,
            num_topics=best_k,
            random_state=seed,
            chunksize=2000,
            passes=LDA_PASSES,
            iterations=LDA_ITERATIONS,
            alpha="asymmetric",
            eta="auto",
            workers=workers,
            eval_every=None,
        )
        seed_terms = top_term_sets(stability_model, best_k)
        matched_scores = []
        for baseline_set in baseline_terms:
            best_jaccard = max(
                len(baseline_set & seed_set) / max(1, len(baseline_set | seed_set))
                for seed_set in seed_terms
            )
            matched_scores.append(best_jaccard)
        stability_rows.append({
            "seed": seed,
            "mean_best_topic_jaccard": float(np.mean(matched_scores)),
            "min_best_topic_jaccard": float(np.min(matched_scores)),
            "max_best_topic_jaccard": float(np.max(matched_scores)),
        })
    stability_summary = pd.DataFrame(stability_rows)
    stability_summary.to_csv(TABLE_DIR / "topic_seed_stability.csv", index=False)
    display(stability_summary)
else:
    print("Seed-stability audit skipped. Set RUN_STABILITY_CHECK = True for a robustness run.")


In [ ]:
# 9. Figure 1: model selection.
fig, ax1 = plt.subplots(figsize=(7.2, 4.2))
ax1.plot(model_selection["topics"], model_selection["coherence_cv"], marker="o", color="#1b6ca8")
ax1.set_xlabel("Number of topics")
ax1.set_ylabel("c_v coherence", color="#1b6ca8")
ax1.tick_params(axis="y", labelcolor="#1b6ca8")
ax1.axvline(best_k, color="#333333", linestyle="--", linewidth=1)

ax2 = ax1.twinx()
ax2.plot(model_selection["topics"], model_selection["perplexity"], marker="s", color="#b44b4b")
ax2.set_ylabel("Perplexity, log scale", color="#b44b4b")
ax2.set_yscale("log")
ax2.tick_params(axis="y", labelcolor="#b44b4b")

fig.suptitle("LDA model selection")
fig.tight_layout()
save_figure(fig, "figure_01_model_selection")
plt.show()


In [ ]:
# 10. Topic-term table and editable labels.
# Leave empty for evidence-based automatic labels. Add manual overrides only after
# inspecting the final Round output workbook and confirming the label matches the keywords.
TOPIC_LABEL_OVERRIDES = {}

def suggest_topic_label(display_keywords, fallback_label):
    keys = {display_key(term) for term in display_keywords}
    joined = " ".join(keys)
    if {"digital payments", "privacy policy", "fraud detection", "gas fees"} & keys:
        return "Digital payments, privacy policy, trust, and fraud detection"
    if {"bitcoin network", "wallet", "cryptocurrency exchange", "consumer protection", "illicit finance"} & keys:
        return "Cryptocurrency networks, wallets, exchanges, and regulatory risk"
    if {"finance", "defi", "scam", "grey economy"} & keys:
        return "Crypto finance, DeFi, scams, and grey-economy risks"
    if {"trust", "governance", "ownership", "metaverse", "blockchain network", "token"} & keys:
        return "Blockchain-based metaverse governance, ownership, and token infrastructure"
    if {"game", "player", "jurisdiction", "legal", "property", "intellectual property"} & keys:
        return "Blockchain gaming, NFT markets, and digital-property law"
    if {"cryptographic key management", "attribute-based encryption", "attribute-based access control", "access control policy", "authentication protocol", "encryption and decryption"} & keys:
        return "Cryptographic access control, authentication, and encryption protocols"
    if {"avatar", "attack", "privacy", "cybersecurity", "detection", "prediction"} & keys:
        return "Avatar privacy, cyberattack detection, and metaverse security"
    if {"smart contract", "protocol", "blockchain", "security"} & keys:
        return "Smart contracts, blockchain protocols, and security mechanisms"
    return fallback_label

def topic_terms(model, topic_index, topn=20):
    return model.show_topic(topic_index, topn=topn)

def readable_term(term):
    display = term.replace("_", " ").strip()
    key = re.sub(r"\s+", " ", display.lower()).strip()
    return DISPLAY_TERM_CANONICAL_MAP.get(key, display)

def display_key(term):
    return re.sub(r"\s+", " ", readable_term(term).lower()).strip()

def display_family_key(term):
    key = display_key(term)
    return DISPLAY_TERM_FAMILY_MAP.get(key, key)

def is_displayable_term(term):
    raw_key = re.sub(r"\s+", " ", term.replace("_", " ").lower()).strip()
    display = readable_term(term)
    key = display_key(term)
    words = key.split()
    raw_words = raw_key.split()
    if not words:
        return False
    if any(fragment in raw_key or fragment in key for fragment in NON_DOMAIN_SUBSTRINGS):
        return False
    if key in NON_DOMAIN_EXACT_TERMS or raw_key in NON_DOMAIN_EXACT_TERMS:
        return False
    if any(word in SOURCE_ARTIFACT_WORDS for word in raw_words + words):
        return False
    if term in DISPLAY_STOPWORDS or display in DISPLAY_STOPWORDS or key in DISPLAY_STOPWORDS:
        return False
    if key in STRICT_DISPLAY_EXCLUDE_TERMS or term in STRICT_DISPLAY_EXCLUDE_TERMS:
        return False
    if any(word in STRICT_TOPIC_NOISE_WORDS for word in words):
        return False
    if all(word in DISPLAY_STOPWORDS for word in words):
        return False
    if len(key) <= 3 and key not in {"nft", "defi", "llm", "dao", "vr", "ar", "ai", "iot"}:
        return False
    if len(words) == 1 and key in NON_DOMAIN_SINGLE_WORDS:
        return False
    if len(words) == 1 and key not in DOMAIN_SINGLE_WORD_WHITELIST:
        return False
    return True

def refined_topic_terms(model, topic_index, topn=None, candidate_n=DISPLAY_CANDIDATE_TERMS):
    raw_terms = model.show_topic(topic_index, topn=candidate_n)
    topic_word_matrix = model.get_topics()
    refined = []
    seen_display = set()
    for raw_term, weight in raw_terms:
        if not is_displayable_term(raw_term):
            continue
        term_id = dictionary.token2id.get(raw_term)
        exclusivity = 0.0
        if term_id is not None:
            exclusivity = float(weight / (topic_word_matrix[:, term_id].sum() + 1e-12))
        phrase_bonus = 1.15 if "_" in raw_term else 1.0
        display_score = float(weight * (0.35 + 0.65 * exclusivity) * phrase_bonus)
        display = readable_term(raw_term)
        key = display_key(display)
        family_key = display_family_key(display)
        if key in seen_display:
            continue
        seen_display.add(key)
        refined.append({
            "raw_term": raw_term,
            "display_term": display,
            "display_key": key,
            "display_family": family_key,
            "weight": float(weight),
            "exclusivity": exclusivity,
            "display_score": display_score,
        })
    refined = sorted(refined, key=lambda row: row["display_score"], reverse=True)
    return refined[:topn] if topn else refined

def choose_display_terms(topic_candidates, topic_number, topn=DISPLAY_KEYWORDS_PER_TOPIC):
    candidates = topic_candidates[topic_number]
    selected = []
    selected_keys = set()
    selected_families = set()
    for row in candidates:
        key = row["display_key"]
        family_key = row.get("display_family", key)
        if key in selected_keys or family_key in selected_families:
            continue
        if ENFORCE_UNIQUE_DISPLAY_KEYWORDS and key in global_display_keyword_keys:
            continue
        selected.append(row)
        selected_keys.add(key)
        selected_families.add(family_key)
        global_display_keyword_keys.add(key)
        if len(selected) >= topn:
            break
    return selected

topic_rows = []
term_weight_rows = []
raw_term_rows = []
topic_candidate_terms = {
    topic_index + 1: refined_topic_terms(best_model, topic_index, candidate_n=DISPLAY_CANDIDATE_TERMS)
    for topic_index in range(best_k)
}
global_display_keyword_keys = set()

for topic_index in range(best_k):
    topic_number = topic_index + 1
    raw_terms = topic_terms(best_model, topic_index, topn=30)
    display_terms = choose_display_terms(topic_candidate_terms, topic_number, topn=DISPLAY_KEYWORDS_PER_TOPIC)
    display_keywords = [row["display_term"] for row in display_terms]
    raw_keywords = [term for term, weight in raw_terms]
    auto_label = " / ".join(display_keywords[:3] or raw_keywords[:3])
    topic_label = TOPIC_LABEL_OVERRIDES.get(topic_number, suggest_topic_label(display_keywords, auto_label))
    topic_rows.append({
        "topic": topic_number,
        "topic_label": topic_label,
        "topic_display": f"T{topic_number}: {topic_label}",
        "auto_label": auto_label,
        "display_keywords": ", ".join(display_keywords[:DISPLAY_KEYWORDS_PER_TOPIC]),
        "raw_model_terms": ", ".join(raw_keywords[:20]),
    })
    for rank, (term, weight) in enumerate(raw_terms, start=1):
        raw_term_rows.append({
            "topic": topic_number,
            "raw_rank": rank,
            "raw_term": term,
            "weight": weight,
        })
    for rank, row in enumerate(display_terms, start=1):
        term_weight_rows.append({
            "topic": topic_number,
            "topic_label": topic_label,
            "topic_display": f"T{topic_number}: {topic_label}",
            "term": row["display_term"],
            "raw_term": row["raw_term"],
            "rank": rank,
            "weight": row["weight"],
            "exclusivity": row["exclusivity"],
            "display_score": row["display_score"],
        })

topic_summary = pd.DataFrame(topic_rows)
term_weights = pd.DataFrame(term_weight_rows)
raw_term_weights = pd.DataFrame(raw_term_rows)
display(topic_summary)
topic_summary.to_csv(TABLE_DIR / "topic_terms_initial.csv", index=False)
term_weights.to_csv(TABLE_DIR / "topic_display_term_weights.csv", index=False)
raw_term_weights.to_csv(TABLE_DIR / "topic_raw_model_terms.csv", index=False)


In [ ]:
# 11. Score chunks and documents.
def dense_topic_vector(model, bow, topic_count):
    vector = np.zeros(topic_count)
    for topic_index, probability in model.get_document_topics(bow, minimum_probability=0):
        vector[topic_index] = probability
    return vector

topic_matrix = np.vstack([dense_topic_vector(best_model, bow, best_k) for bow in tqdm(all_bows, desc="Scoring chunks")])
topic_columns = [f"topic_{index + 1:02d}" for index in range(best_k)]

chunk_topics = scored_chunks.drop(columns=["tokens"]).copy()
chunk_topics = pd.concat([chunk_topics, pd.DataFrame(topic_matrix, columns=topic_columns)], axis=1)
chunk_topics["dominant_topic"] = topic_matrix.argmax(axis=1) + 1
chunk_topics["dominant_topic_probability"] = topic_matrix.max(axis=1)
chunk_topics.to_csv(TABLE_DIR / "chunk_topic_matrix.csv", index=False)

doc_topic = chunk_topics.groupby(["doc_id", "filename"], as_index=False)[topic_columns].mean()
doc_topic["dominant_topic"] = doc_topic[topic_columns].values.argmax(axis=1) + 1
doc_topic["dominant_topic_probability"] = doc_topic[topic_columns].max(axis=1)
doc_topic.to_csv(TABLE_DIR / "document_topic_matrix.csv", index=False)

prevalence = doc_topic[topic_columns].mean().rename_axis("topic_column").reset_index(name="document_normalised_prevalence")
prevalence["topic"] = prevalence["topic_column"].str.extract(r"(\d+)").astype(int)
topic_summary = topic_summary.merge(prevalence[["topic", "document_normalised_prevalence"]], on="topic", how="left")
topic_summary = topic_summary.sort_values("document_normalised_prevalence", ascending=False)
topic_summary.to_csv(TABLE_DIR / "topic_summary.csv", index=False)

exemplar_rows = []
for topic_number, column in enumerate(topic_columns, start=1):
    top_chunks = chunk_topics.sort_values(column, ascending=False).head(8)
    for row in top_chunks.itertuples(index=False):
        exemplar_rows.append({
            "topic": topic_number,
            "topic_probability": getattr(row, column),
            "chunk_id": row.chunk_id,
            "filename": row.filename,
            "chunk_number": row.chunk_number,
            "snippet_tokens": row.snippet,
        })
exemplars = pd.DataFrame(exemplar_rows)
exemplars.to_csv(TABLE_DIR / "exemplar_chunks_by_topic.csv", index=False)

exemplar_brief = exemplars.groupby("topic", as_index=False).agg(
    exemplar_filenames=("filename", lambda values: " | ".join(dict.fromkeys(list(values)[:5]))),
    exemplar_snippets=("snippet_tokens", lambda values: " || ".join(list(values)[:3])),
)
topic_labelling_worksheet = topic_summary.merge(exemplar_brief, on="topic", how="left")
topic_labelling_worksheet["final_label"] = topic_labelling_worksheet["topic_label"]
topic_labelling_worksheet["researcher_note"] = ""
topic_labelling_worksheet.to_csv(TABLE_DIR / "topic_labelling_worksheet.csv", index=False)


keyword_lists = term_weights.sort_values(["topic", "rank"]).groupby("topic").agg(
    normalised_keywords=("term", lambda values: "; ".join(values)),
    top_10_normalised_keywords=("term", lambda values: "; ".join(list(values)[:10])),
).reset_index()

topic_modelling_table = topic_labelling_worksheet.drop(columns=["display_keywords"], errors="ignore").merge(
    keyword_lists,
    on="topic",
    how="left",
)
topic_modelling_table = topic_modelling_table[[
    "topic",
    "topic_label",
    "document_normalised_prevalence",
    "normalised_keywords",
    "top_10_normalised_keywords",
    "raw_model_terms",
    "exemplar_filenames",
]]
topic_modelling_table["document_normalised_prevalence_percent"] = (
    topic_modelling_table["document_normalised_prevalence"] * 100
).round(2)
topic_modelling_table = topic_modelling_table[[
    "topic",
    "topic_label",
    "document_normalised_prevalence_percent",
    "normalised_keywords",
    "top_10_normalised_keywords",
    "raw_model_terms",
    "exemplar_filenames",
]].rename(columns={
    "topic": "Topic",
    "topic_label": "Interpretive topic label",
    "document_normalised_prevalence_percent": "Document-normalised prevalence (%)",
    "normalised_keywords": "Normalised keywords",
    "top_10_normalised_keywords": "Top 10 normalised keywords",
    "raw_model_terms": "Raw model terms retained for audit",
    "exemplar_filenames": "Representative PDFs",
})
topic_modelling_table_with_raw_audit = topic_modelling_table.copy()
topic_modelling_table = topic_modelling_table.drop(columns=["Raw model terms retained for audit"], errors="ignore")

topic_modelling_table.to_csv(TABLE_DIR / "topic_modelling_table_normalised_keywords.csv", index=False)
topic_modelling_table.to_excel(TABLE_DIR / "topic_modelling_table_normalised_keywords.xlsx", index=False)
topic_modelling_table.to_html(TABLE_DIR / "topic_modelling_table_normalised_keywords.html", index=False, escape=False)
topic_modelling_table_with_raw_audit.to_csv(TABLE_DIR / "topic_modelling_table_with_raw_audit.csv", index=False)
topic_modelling_table_with_raw_audit.to_excel(TABLE_DIR / "topic_modelling_table_with_raw_audit.xlsx", index=False)
try:
    (TABLE_DIR / "topic_modelling_table_normalised_keywords.md").write_text(
        topic_modelling_table.to_markdown(index=False),
        encoding="utf-8",
    )
except Exception as exc:
    print(f"Markdown topic table skipped because optional formatter is unavailable: {exc}")
display(topic_modelling_table)
print(f"Formatted publication topic table written to: {TABLE_DIR / 'topic_modelling_table_normalised_keywords.xlsx'}")
print(f"Audit topic table with raw terms written to: {TABLE_DIR / 'topic_modelling_table_with_raw_audit.xlsx'}")

display(topic_summary)
display(doc_topic.sort_values("dominant_topic_probability", ascending=False).head(12))


In [ ]:
# 11b. Publication diagnostics: diversity, entropy, and topic similarity.
def safe_entropy(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities / probabilities.sum()
    return float(-(probabilities * np.log(probabilities + 1e-12)).sum())

topic_word_matrix = best_model.get_topics()
topn = min(TOP_TERMS_FOR_DIAGNOSTICS, len(dictionary))
topic_term_sets = []
for topic_index in range(best_k):
    top_ids = topic_word_matrix[topic_index].argsort()[::-1][:topn]
    topic_term_sets.append({dictionary[int(term_id)] for term_id in top_ids})

unique_top_terms = set().union(*topic_term_sets) if topic_term_sets else set()
topic_diversity = len(unique_top_terms) / max(1, best_k * topn)

doc_prob_matrix = doc_topic[topic_columns].to_numpy()
document_entropy = np.apply_along_axis(safe_entropy, 1, doc_prob_matrix) / np.log(best_k)
document_topic_diagnostics = doc_topic[["doc_id", "filename", "dominant_topic", "dominant_topic_probability"]].copy()
document_topic_diagnostics["normalised_topic_entropy"] = document_entropy
document_topic_diagnostics.to_csv(TABLE_DIR / "document_topic_diagnostics.csv", index=False)

def jensen_shannon_distance(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    divergence = 0.5 * np.sum(p * np.log2((p + 1e-12) / (m + 1e-12)))
    divergence += 0.5 * np.sum(q * np.log2((q + 1e-12) / (m + 1e-12)))
    return float(np.sqrt(max(divergence, 0)))

topic_similarity = np.ones((best_k, best_k))
for i in range(best_k):
    for j in range(best_k):
        topic_similarity[i, j] = 1 - jensen_shannon_distance(topic_word_matrix[i], topic_word_matrix[j])

topic_similarity_df = pd.DataFrame(
    topic_similarity,
    index=[f"T{index + 1}" for index in range(best_k)],
    columns=[f"T{index + 1}" for index in range(best_k)],
)
topic_similarity_df.to_csv(TABLE_DIR / "topic_similarity_matrix.csv")

topic_quality_metrics = pd.DataFrame([{
    "selected_topics": best_k,
    "selected_coherence_cv": float(selected_model_metrics["coherence_cv"].iloc[0]),
    "selected_log_likelihood_bound": float(selected_model_metrics["log_likelihood_bound"].iloc[0]),
    "selected_per_word_log_likelihood_bound": float(selected_model_metrics["per_word_log_likelihood_bound"].iloc[0]),
    "selected_log_perplexity": float(selected_model_metrics["log_perplexity"].iloc[0]),
    "selected_perplexity": float(selected_model_metrics["perplexity"].iloc[0]),
    "topic_diversity_top_terms": topic_diversity,
    "mean_document_entropy": float(document_topic_diagnostics["normalised_topic_entropy"].mean()),
    "median_document_entropy": float(document_topic_diagnostics["normalised_topic_entropy"].median()),
    "mean_dominant_topic_probability": float(doc_topic["dominant_topic_probability"].mean()),
    "min_dominant_topic_probability": float(doc_topic["dominant_topic_probability"].min()),
}])
topic_quality_metrics.to_csv(TABLE_DIR / "topic_quality_metrics.csv", index=False)

display(topic_quality_metrics)
display(document_topic_diagnostics.sort_values("normalised_topic_entropy", ascending=False).head(10))


In [ ]:
# 12. Figure 2: document-normalised topic prevalence.
plot_df = topic_summary.sort_values("document_normalised_prevalence", ascending=True).copy()
plot_df["topic_display"] = plot_df.apply(lambda row: f"T{int(row.topic)}: {row.topic_label}", axis=1)

fig_height = max(4.2, 0.45 * len(plot_df))
fig, ax = plt.subplots(figsize=(8.2, fig_height))
sns.barplot(
    data=plot_df,
    x="document_normalised_prevalence",
    y="topic_display",
    ax=ax,
    color="#2f7f7b",
)
ax.set_xlabel("Mean topic proportion across PDFs")
ax.set_ylabel("")
ax.set_title("Document-normalised topic prevalence")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
fig.tight_layout()
save_figure(fig, "figure_02_topic_prevalence")
plt.show()


In [ ]:
# 13. Figure 3: top 20 normalised keywords across all documents.
top_keyword_plot = top20_normalised_keywords.sort_values(["document_frequency", "token_frequency"], ascending=True).copy()

fig, ax = plt.subplots(figsize=(8.6, max(5.2, 0.35 * len(top_keyword_plot))))
sns.barplot(
    data=top_keyword_plot,
    x="document_frequency",
    y="normalised_keyword",
    color="#1f77b4",
    ax=ax,
)
ax.set_xlabel("Number of PDFs containing keyword")
ax.set_ylabel("")
ax.set_title("Top 20 Normalised Keywords Across Documents")
for container in ax.containers:
    ax.bar_label(container, labels=[str(int(value)) for value in top_keyword_plot["document_frequency"]], padding=3, fontsize=8)
fig.tight_layout()
save_figure(fig, "figure_03_top20_normalised_keywords")
plt.show()


In [ ]:
# 13. Figure 3: topic-keyword heatmap.
top_terms = (
    term_weights[term_weights["rank"] <= 8]
    .groupby("term")["weight"]
    .max()
    .sort_values(ascending=False)
    .head(35)
    .index
)
heatmap_data = (
    term_weights[term_weights["term"].isin(top_terms)]
    .pivot_table(index="topic_display", columns="term", values="weight", fill_value=0)
)
heatmap_data = heatmap_data.loc[topic_summary["topic_display"]]
heatmap_data = heatmap_data[top_terms]

fig_width = max(9.0, 0.32 * len(heatmap_data.columns))
fig_height = max(4.5, 0.45 * len(heatmap_data))
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
sns.heatmap(heatmap_data, cmap="viridis", linewidths=0.3, linecolor="white", ax=ax)
ax.set_xlabel("Keyword")
ax.set_ylabel("Topic")
ax.set_title("Topic-keyword weights")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
save_figure(fig, "figure_03_topic_keyword_heatmap")
plt.show()


In [ ]:
# 14b. Figure 4b: normalised keyword co-occurrence network.
if USE_CONTROLLED_DOMAIN_KEYWORDS_FOR_TOP20 and not top20_normalised_keywords.empty:
    network_keyword_set = set(top20_normalised_keywords["normalised_keyword"])
else:
    network_keyword_set = set(normalised_keyword_frequencies.head(COOCCURRENCE_TOP_KEYWORDS)["normalised_keyword"])
edge_counter = Counter()
node_chunk_counter = Counter()

for row in chunks.itertuples(index=False):
    chunk_terms = sorted({
        keyword_display_key(token)
        for token in row.tokens
        if keyword_is_displayable(token) and keyword_display_key(token) in network_keyword_set
    })
    node_chunk_counter.update(chunk_terms)
    for left, right in combinations(chunk_terms, 2):
        edge_counter[(left, right)] += 1

cooccurrence_edges = pd.DataFrame([
    {"source": left, "target": right, "chunk_cooccurrence_count": int(weight)}
    for (left, right), weight in edge_counter.items()
    if weight >= COOCCURRENCE_MIN_EDGE_WEIGHT
]).sort_values("chunk_cooccurrence_count", ascending=False)

cooccurrence_nodes = normalised_keyword_frequencies[
    normalised_keyword_frequencies["normalised_keyword"].isin(network_keyword_set)
].copy()
cooccurrence_nodes["chunk_frequency"] = cooccurrence_nodes["normalised_keyword"].map(node_chunk_counter).fillna(0).astype(int)
cooccurrence_nodes.to_csv(TABLE_DIR / "normalised_keyword_cooccurrence_nodes.csv", index=False)
cooccurrence_edges.to_csv(TABLE_DIR / "normalised_keyword_cooccurrence_edges.csv", index=False)

if cooccurrence_edges.empty:
    print("No co-occurrence edges met the minimum threshold. Lower COOCCURRENCE_MIN_EDGE_WEIGHT and rerun this cell.")
else:
    graph = nx.Graph()
    for row in cooccurrence_nodes.itertuples(index=False):
        graph.add_node(
            row.normalised_keyword,
            document_frequency=int(row.document_frequency),
            token_frequency=int(row.token_frequency),
            chunk_frequency=int(row.chunk_frequency),
        )
    for row in cooccurrence_edges.itertuples(index=False):
        graph.add_edge(row.source, row.target, weight=int(row.chunk_cooccurrence_count))
    graph.remove_nodes_from([node for node, degree in dict(graph.degree()).items() if degree == 0])

    fig, ax = plt.subplots(figsize=(10.5, 8.5))
    positions = nx.spring_layout(graph, seed=RANDOM_STATE, k=0.7 / max(1, math.sqrt(graph.number_of_nodes())))
    node_values = np.array([graph.nodes[node].get("document_frequency", 1) for node in graph.nodes()], dtype=float)
    node_sizes = 90 + 780 * (node_values / max(1, node_values.max()))
    edge_values = np.array([graph.edges[edge].get("weight", 1) for edge in graph.edges()], dtype=float)
    edge_widths = 0.4 + 3.2 * (edge_values / max(1, edge_values.max()))

    nx.draw_networkx_edges(graph, positions, width=edge_widths, alpha=0.28, edge_color="#4d4d4d", ax=ax)
    nx.draw_networkx_nodes(
        graph,
        positions,
        node_size=node_sizes,
        node_color=node_values,
        cmap="viridis",
        edgecolors="white",
        linewidths=0.8,
        alpha=0.95,
        ax=ax,
    )
    label_candidates = sorted(
        graph.nodes(),
        key=lambda node: (graph.degree(node), graph.nodes[node].get("document_frequency", 0)),
        reverse=True,
    )[:COOCCURRENCE_LABEL_COUNT]
    nx.draw_networkx_labels(graph, positions, labels={node: node for node in label_candidates}, font_size=8, ax=ax)
    ax.set_title("Normalised Keyword Co-occurrence Network")
    ax.set_axis_off()
    sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(vmin=node_values.min(), vmax=node_values.max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.01)
    cbar.set_label("PDF document frequency")
    fig.tight_layout()
    save_figure(fig, "figure_04b_normalised_keyword_cooccurrence_network")
    plt.show()


In [ ]:
# 14. Figure 4: document-topic heatmap.
doc_plot = doc_topic.copy()
doc_plot["short_title"] = doc_plot["filename"].str.replace(r"\.pdf$", "", regex=True).str.slice(0, 72)
doc_plot = doc_plot.sort_values(["dominant_topic", "dominant_topic_probability"], ascending=[True, False])
doc_heat = doc_plot.set_index("short_title")[topic_columns]
doc_heat.columns = [f"T{index + 1}" for index in range(best_k)]

fig_height = min(24, max(8, 0.28 * len(doc_heat)))
fig_width = max(7.5, 0.55 * best_k)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
sns.heatmap(doc_heat, cmap="mako", linewidths=0.2, linecolor="white", ax=ax, cbar_kws={"label": "Topic proportion"})
ax.set_xlabel("Topic")
ax.set_ylabel("PDF")
ax.set_title("Document-topic matrix")
fig.tight_layout()
save_figure(fig, "figure_04_document_topic_heatmap")
plt.show()


In [ ]:
# 15. Figure 5: topic similarity heatmap.
fig, ax = plt.subplots(figsize=(max(5.5, best_k * 0.55), max(4.8, best_k * 0.45)))
sns.heatmap(
    topic_similarity_df,
    cmap="crest",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "1 - Jensen-Shannon distance"},
)
ax.set_title("Topic similarity")
ax.set_xlabel("Topic")
ax.set_ylabel("Topic")
fig.tight_layout()
save_figure(fig, "figure_05_topic_similarity_heatmap")
plt.show()


In [ ]:
# 16. Figure 6: labelled topic word clouds.
try:
    from wordcloud import WordCloud

    def format_prevalence(value):
        if pd.isna(value):
            return "n/a"
        return f"{float(value) * 100:.2f}%"

    def wrap_for_title(value, width=38):
        return textwrap.fill(str(value), width=width, break_long_words=False)

    wordcloud_label_rows = []
    cols = min(3, best_k)
    rows = math.ceil(best_k / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6.0, rows * 4.8))
    axes = np.array(axes).reshape(-1)
    topic_summary_lookup = topic_summary.set_index("topic")

    for topic_index in range(best_k):
        topic_number = topic_index + 1
        summary_row = topic_summary_lookup.loc[topic_number]
        topic_label = str(summary_row["topic_label"])
        prevalence_text = format_prevalence(summary_row.get("document_normalised_prevalence", np.nan))
        cloud_terms = term_weights[term_weights["topic"] == topic_number].sort_values("rank")
        frequencies = {
            str(term): float(score)
            for term, score in zip(cloud_terms["term"], cloud_terms["display_score"])
            if pd.notna(score) and float(score) > 0
        }
        key_terms = [str(term) for term in cloud_terms["term"].head(5).tolist()]

        if not frequencies:
            raw_terms = best_model.show_topic(topic_index, topn=60)
            frequencies = {readable_term(term): float(weight) for term, weight in raw_terms if float(weight) > 0}
            key_terms = [readable_term(term) for term, _ in raw_terms[:5]]

        cloud = WordCloud(
            width=1400,
            height=900,
            background_color="white",
            colormap="viridis",
            prefer_horizontal=0.94,
            random_state=RANDOM_STATE,
            max_words=45,
            min_font_size=8,
            relative_scaling=0.40,
            collocations=False,
            normalize_plurals=False,
        ).generate_from_frequencies(frequencies)

        keyword_title = textwrap.fill(", ".join(key_terms[:5]), width=52, break_long_words=False)
        axes[topic_index].imshow(cloud, interpolation="bilinear")
        axes[topic_index].set_title(
            f"Topic {topic_number}: {wrap_for_title(topic_label)}\n"
            f"Prevalence: {prevalence_text}\n"
            f"Key terms: {keyword_title}",
            fontsize=10.5,
            fontweight="bold",
            loc="left",
            pad=12,
        )
        axes[topic_index].axis("off")

        wordcloud_label_rows.append({
            "topic": topic_number,
            "wordcloud_label": f"Topic {topic_number}: {topic_label}",
            "document_normalised_prevalence_percent": prevalence_text,
            "wordcloud_key_terms": ", ".join(key_terms[:8]),
            "interpretation_note": (
                "Use this label with Figure 6; key terms are the strongest "
                "publication-facing display terms used in the word cloud."
            ),
        })

    for empty_axis in axes[best_k:]:
        empty_axis.axis("off")

    fig.suptitle(
        "Topic word clouds with interpretive labels",
        fontsize=15,
        fontweight="bold",
        y=0.995,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94], h_pad=3.2, w_pad=1.2)
    wordcloud_label_guide = pd.DataFrame(wordcloud_label_rows)
    wordcloud_label_guide.to_csv(TABLE_DIR / "topic_wordcloud_label_guide.csv", index=False)
    save_figure(fig, "figure_06_topic_wordclouds_labelled")
    save_figure(fig, "figure_06_topic_wordclouds")
    plt.show()
    print(f"Word-cloud label guide written to: {TABLE_DIR / 'topic_wordcloud_label_guide.csv'}")
except Exception as exc:
    print(f"Word cloud generation skipped: {exc}")


In [ ]:
# 17. Interactive pyLDAvis view.
try:
    import pyLDAvis
    import pyLDAvis.gensim_models as gensimvis

    vis = gensimvis.prepare(best_model, model_bows, dictionary, sort_topics=False)
    html_path = FIGURE_DIR / "figure_07_pyldavis.html"
    pyLDAvis.save_html(vis, str(html_path))
    print(f"Interactive LDA visualisation written to: {html_path}")
    display(HTML(html_path.read_text(encoding="utf-8")))
except Exception as exc:
    print(f"pyLDAvis generation skipped: {exc}")


## Sensitivity checks for publication use

Before finalising results for a manuscript, run at least one sensitivity check:

- Add or remove conservative domain stopwords and compare topic labels.
- Change `TOPIC_RANGE`, `CHUNK_SIZE`, or `MAX_CHUNKS_PER_DOCUMENT_FOR_MODEL`.
- Compare the coherence-optimal topic count with a nearby, more interpretable topic count.
- Inspect long PDFs separately if they are edited volumes, books, theses, or proceedings rather than journal articles.

Keep the outputs from the final selected run and report the settings in the methods note generated below.


In [ ]:
# 18. Methods note and reproducibility metadata.
methods_note = f'''
## Topic modelling methods note

The corpus comprised {len(manifest)} PDF files, of which {manifest["included_for_modelling"].sum()} were included in the final modelling set after applying the duplicate-text rule. Text was extracted with pypdf, then lower-cased, cleaned of URLs, DOI strings, numeric strings, and common publication boilerplate. Reference sections were {'removed' if REMOVE_REFERENCES else 'retained'} using a conservative heading-based rule. Exact duplicate extracted texts were {'excluded after the first copy' if DROP_DUPLICATE_TEXTS else 'retained'} for modelling. Tokens shorter than {MIN_TOKEN_LENGTH} characters and English stopwords were removed. Remaining tokens were lemmatised with WordNet and common bigrams/trigrams were detected using Gensim Phrases.

Documents were split into overlapping chunks of {CHUNK_SIZE} tokens with {CHUNK_OVERLAP} token overlap; chunks shorter than {MIN_CHUNK_TOKENS} tokens were excluded. LDA models were estimated in Gensim using `LdaMulticore` with asymmetric alpha, automatic eta, {LDA_PASSES} passes, {LDA_ITERATIONS} iterations, and random seed {RANDOM_STATE}. Candidate topic counts were evaluated over {list(TOPIC_RANGE)} using c_v coherence, the Gensim variational log-likelihood bound, per-word log-likelihood bound, log perplexity, and perplexity. The selected model used {best_k} topics, based on {topic_selection_decision}. The topic-count conclusion was exported to `topic_count_conclusion.md`.

Topic prevalence was calculated by averaging topic proportions across chunks within each PDF, then averaging those PDF-level proportions across the corpus. This document-normalised estimate prevents long PDFs from receiving disproportionate influence in prevalence reporting. Topic quality diagnostics included top-term diversity, document-topic entropy, dominant-topic concentration, and pairwise topic similarity based on Jensen-Shannon distance. Topic labels were finalised through researcher interpretation of top terms, exemplar chunks, and the interactive LDA visualisation. The publication-facing topic-keyword table uses a strict display layer that normalises domain variants, repairs common PDF word-splitting artifacts, removes publisher/OCR/non-English fragments, and avoids generic labels where possible; raw model terms are retained separately for auditability.
'''

metadata = {
    "pdf_count": int(len(manifest)),
    "readable_pdf_count": int((manifest["read_status"] == "ok").sum()),
    "included_pdf_count": int(manifest["included_for_modelling"].sum()),
    "duplicate_text_count": int(manifest["duplicate_text_flag"].sum()),
    "chunk_count": int(len(chunks)),
    "model_training_chunk_count": int(len(model_chunks)),
    "scored_chunk_count": int(len(scored_chunks)),
    "vocabulary_size": int(len(dictionary)),
    "selected_topic_count": int(best_k),
    "topic_selection_decision": topic_selection_decision,
    "force_topic_count": FORCE_TOPIC_COUNT,
    "selected_coherence_cv": float(selected_model_metrics["coherence_cv"].iloc[0]),
    "selected_log_likelihood_bound": float(selected_model_metrics["log_likelihood_bound"].iloc[0]),
    "selected_per_word_log_likelihood_bound": float(selected_model_metrics["per_word_log_likelihood_bound"].iloc[0]),
    "selected_log_perplexity": float(selected_model_metrics["log_perplexity"].iloc[0]),
    "selected_perplexity": float(selected_model_metrics["perplexity"].iloc[0]),
    "topic_diversity_top_terms": float(topic_quality_metrics["topic_diversity_top_terms"].iloc[0]),
    "mean_document_entropy": float(topic_quality_metrics["mean_document_entropy"].iloc[0]),
    "random_state": RANDOM_STATE,
    "topic_range": list(TOPIC_RANGE),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "max_chunks_per_document_for_model": MAX_CHUNKS_PER_DOCUMENT_FOR_MODEL,
    "run_stability_check": RUN_STABILITY_CHECK,
}

(OUTPUT_DIR / "methods_note.md").write_text(methods_note, encoding="utf-8")
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

topic_lines = []
for row in topic_summary.sort_values("document_normalised_prevalence", ascending=False).itertuples(index=False):
    topic_lines.append(
        f"- Topic {int(row.topic)} ({row.topic_label}): "
        f"{row.document_normalised_prevalence:.1%} document-normalised prevalence; "
        f"keywords: {row.display_keywords}."
    )

results_text = "## Results text template\n\n"
results_text += (
    f"Topic-count conclusion: choose {best_k} topics for the main LDA model. The selection is based on c_v coherence, likelihood/perplexity diagnostics, and interpretability checks. "
    "Document-normalised topic prevalence was used because PDF lengths varied substantially across the corpus. "
    "The highest-prevalence topics were:\n\n"
)
results_text += "\n".join(topic_lines)
results_text += (
    "\n\nThese labels are draft interpretive labels. Final labels should be confirmed by reviewing "
    "`topic_labelling_worksheet.csv`, exemplar chunks, and the interactive pyLDAvis output."
)

figure_captions = '''## Figure captions

Figure 1. LDA model selection across candidate topic counts, showing c_v coherence and perplexity. The dashed line marks the coherence-selected topic count. The companion model-selection table reports log-likelihood bound, per-word log-likelihood bound, log perplexity, and perplexity for each candidate.

Figure 2. Document-normalised topic prevalence. Values are mean topic proportions after averaging chunk-level topic weights within each PDF and then across PDFs.

Figure 3. Topic-keyword weight heatmap using publication-facing display keywords. Raw model terms are retained separately for auditability.

Figure 4. Document-topic heatmap showing the topic mixture for each PDF.

Figure 5. Topic similarity heatmap based on one minus Jensen-Shannon distance between topic-word distributions.

Figure 6. Labelled topic word clouds using publication-facing display keywords weighted by topic probability and term exclusivity. Each panel reports the interpretive topic label, document-normalised prevalence, and the five strongest display terms used to guide interpretation.

Figure 7. Interactive pyLDAvis visualisation for topic inspection.

Additional figure. Top 20 normalised keywords across all documents, ranked by the number of PDFs in which each keyword appears.

Additional figure. Normalised keyword co-occurrence network. Node colour and size reflect PDF document frequency; edge weight reflects chunk-level co-occurrence frequency.
'''

(OUTPUT_DIR / "results_text_template.md").write_text(results_text, encoding="utf-8")
(OUTPUT_DIR / "figure_captions.md").write_text(figure_captions, encoding="utf-8")
print(methods_note)
print(f"Methods note written to: {OUTPUT_DIR / 'methods_note.md'}")


In [ ]:
# 19. Build a publication workbook.
workbook_path = TABLE_DIR / "lda_publication_tables.xlsx"
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    manifest.to_excel(writer, sheet_name="corpus_manifest", index=False)
    token_audit.to_excel(writer, sheet_name="tokenisation_audit", index=False)
    text_export_manifest.to_excel(writer, sheet_name="text_export", index=False)
    normalization_map.to_excel(writer, sheet_name="normalisation_map", index=False)
    normalised_keyword_frequencies.to_excel(writer, sheet_name="normalised_keywords", index=False)
    pd.DataFrame({"keyword_reporting_stopword": sorted(keyword_reporting_stopwords)}).to_excel(writer, sheet_name="keyword_stopwords", index=False)
    top20_normalised_keywords.to_excel(writer, sheet_name="top20_keywords", index=False)
    if "domain_keyword_frequencies" in globals():
        domain_keyword_frequencies.to_excel(writer, sheet_name="domain_keywords_full", index=False)
    model_selection.to_excel(writer, sheet_name="model_selection", index=False)
    selected_model_metrics.to_excel(writer, sheet_name="selected_model", index=False)
    topic_count_recommendation.to_excel(writer, sheet_name="topic_count_choice", index=False)
    topic_summary.to_excel(writer, sheet_name="topic_summary", index=False)
    topic_labelling_worksheet.to_excel(writer, sheet_name="topic_labels", index=False)
    topic_modelling_table.to_excel(writer, sheet_name="topic_table_final", index=False)
    doc_topic.to_excel(writer, sheet_name="document_topics", index=False)
    document_topic_diagnostics.to_excel(writer, sheet_name="document_diagnostics", index=False)
    topic_quality_metrics.to_excel(writer, sheet_name="quality_metrics", index=False)
    topic_similarity_df.to_excel(writer, sheet_name="topic_similarity")
    exemplars.to_excel(writer, sheet_name="exemplar_chunks", index=False)
    term_weights.to_excel(writer, sheet_name="display_term_weights", index=False)
    raw_term_weights.to_excel(writer, sheet_name="raw_model_terms", index=False)
    if "wordcloud_label_guide" in globals():
        wordcloud_label_guide.to_excel(writer, sheet_name="wordcloud_labels", index=False)
    if "cooccurrence_nodes" in globals():
        cooccurrence_nodes.to_excel(writer, sheet_name="cooccur_nodes", index=False)
    if "cooccurrence_edges" in globals():
        cooccurrence_edges.to_excel(writer, sheet_name="cooccur_edges", index=False)
    if not stability_summary.empty:
        stability_summary.to_excel(writer, sheet_name="seed_stability", index=False)

print(f"Publication workbook written to: {workbook_path}")


In [ ]:
# 20. Build an output manifest.
output_records = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        output_records.append({
            "relative_path": str(path.relative_to(OUTPUT_DIR)),
            "size_bytes": path.stat().st_size,
            "category": path.parent.name,
        })

output_manifest = pd.DataFrame(output_records)
output_manifest.to_csv(TABLE_DIR / "output_manifest.csv", index=False)
(OUTPUT_DIR / "output_manifest.json").write_text(
    json.dumps(output_records, indent=2),
    encoding="utf-8",
)
display(output_manifest)


In [ ]:
# 21. Export all outputs.
zip_path = shutil.make_archive("/content/metaverse_crypto_lda_outputs", "zip", OUTPUT_DIR)
print(f"Output archive ready: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download helper is available only in Colab. Retrieve the ZIP from /content manually.")
